<a href="https://colab.research.google.com/github/tannishgarg01/ACT-104-Week-2-/blob/main/ACT_104_Week_2_(Tannish_Garg).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 — Multimodel Chat Playground

**Name:** Tannish Garg

This notebook uses **OpenRouter** as a single gateway to compare multiple LLMs on the same three tasks:
1. Creative Writing
2. Analytical Reasoning
3. Instruction Following

The same prompt is sent to every selected model so that the model is the main variable in the comparison.

> **Security:** The API key is entered at runtime with `getpass` and is never stored in the notebook.


In [5]:
# Install the OpenAI-compatible client used by OpenRouter.
!pip -q install openai


In [6]:
from getpass import getpass
from openai import OpenAI
import json
import time
import re
from IPython.display import display, Markdown

# Enter your key when the cell runs. Do NOT type it into notebook source code.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Models from different providers, routed through the same OpenRouter endpoint.
OPENAI_MODEL = "openai/gpt-4o-mini"
GEMINI_MODEL = "google/gemini-2.5-flash"
CLAUDE_MODEL = "anthropic/claude-sonnet-4"

print("API key loaded:", bool(OPENROUTER_API_KEY))
print("Models configured:", OPENAI_MODEL, GEMINI_MODEL, CLAUDE_MODEL)


Enter your OpenRouter API key: ··········
API key loaded: True
Models configured: openai/gpt-4o-mini google/gemini-2.5-flash anthropic/claude-sonnet-4


## Student Functions

The assignment expects three functions. All three use the same OpenRouter API key and endpoint, but each function routes the request to a different model.

The functions return **plain text only** and do not print inside the function.


In [7]:
def _call_openrouter(prompt: str, model: str, temperature: float = 0.3, max_tokens: int = 250) -> str:
    """Send one prompt to one OpenRouter model and return its text response."""
    client = OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
    )

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content or ""


def call_openai(prompt: str) -> str:
    """Call the selected OpenAI model through OpenRouter."""
    return _call_openrouter(prompt, OPENAI_MODEL)


def call_gemini(prompt: str) -> str:
    """Call the selected Gemini model through OpenRouter."""
    return _call_openrouter(prompt, GEMINI_MODEL)


def call_claude(prompt: str) -> str:
    """Call the selected Claude model through OpenRouter."""
    return _call_openrouter(prompt, CLAUDE_MODEL)


In [8]:
# Quick sanity check
print("OpenAI:", call_openai("Say hi in 3 words."))
print("Gemini:", call_gemini("Say hi in 3 words."))
print("Claude:", call_claude("Say hi in 3 words."))


OpenAI: Hello there, friend!
Gemini: Hello there!
Claude: Hi there friend!


## 1. Creative Writing

**Prompt:** Write a 4-line poem about learning by testing code.

Evaluation focus:
- Creativity
- Tone
- Originality
- Whether the model follows the 4-line constraint


In [9]:
CREATIVE_PROMPT = "Write a 4-line poem about learning by testing code."


## 2. Analytical Reasoning

**Prompt:** Summarize the following idea in one sentence and keep it within 40 words.

Evaluation focus:
- Accuracy
- Conciseness
- Clarity
- Sentence/word-count compliance


In [10]:
ANALYTICAL_PROMPT = (
    "Summarize in one sentence (<= 40 words): "
    "'Using multiple LLM providers helps compare creativity, formatting reliability, and speed, "
    "but adds integration overhead and testing complexity.'"
)


## 3. Instruction Following

**Prompt:** Return only valid JSON with an exact schema.

Evaluation focus:
- Valid JSON
- Exact key names
- Exactly two list items
- Correct data types
- No Markdown fences or extra explanation


In [11]:
INSTRUCTION_PROMPT = (
    "Return ONLY valid JSON (no backticks). The JSON must be an object with key 'todos' "
    "which is a list of EXACTLY 2 items. Each item must be an object with keys 'task' (string) "
    "and 'done' (boolean)."
)


## Comparative Playtest

The function below sends the **same prompt** to all three models and records response time and output.

This lets us compare model behavior while keeping the task constant.


In [12]:
MODEL_FUNCTIONS = {
    "OpenAI GPT-4o-mini": call_openai,
    "Google Gemini 2.5 Flash": call_gemini,
    "Anthropic Claude 3.5 Sonnet": call_claude,
}

def run_comparison(prompt: str):
    results = {}

    for model_name, function in MODEL_FUNCTIONS.items():
        start = time.perf_counter()
        try:
            output = function(prompt)
            error = None
        except Exception as exc:
            output = ""
            error = str(exc)

        elapsed = time.perf_counter() - start

        results[model_name] = {
            "output": output,
            "seconds": round(elapsed, 3),
            "error": error,
        }

    return results


In [13]:
creative_results = run_comparison(CREATIVE_PROMPT)
analytical_results = run_comparison(ANALYTICAL_PROMPT)
instruction_results = run_comparison(INSTRUCTION_PROMPT)


In [14]:
def show_results(title, results):
    print("=" * 80)
    print(title)
    print("=" * 80)

    for model, result in results.items():
        print(f"\nMODEL: {model}")
        print(f"Response time: {result['seconds']} seconds")

        if result["error"]:
            print("ERROR:", result["error"])
        else:
            print("OUTPUT:")
            print(result["output"])

show_results("CREATIVE WRITING RESULTS", creative_results)
show_results("ANALYTICAL REASONING RESULTS", analytical_results)
show_results("INSTRUCTION FOLLOWING RESULTS", instruction_results)


CREATIVE WRITING RESULTS

MODEL: OpenAI GPT-4o-mini
Response time: 1.649 seconds
OUTPUT:
In lines of code, we weave our dreams,  
Through trials and errors, knowledge gleams.  
Each bug a lesson, each fix a gain,  
In the world of testing, wisdom reigns.

MODEL: Google Gemini 2.5 Flash
Response time: 0.977 seconds
OUTPUT:
Bugs appear, a tangled mess,
With each new test, we learn to stress.
But step by step, the logic gleams,
Code refined, fulfilling dreams.

MODEL: Anthropic Claude 3.5 Sonnet
Response time: 2.336 seconds
OUTPUT:
Here's a 4-line poem about learning by testing code:

Each bug I find becomes my teacher,
Red errors guide me to the truth,
Through trial and failure, I grow wiser—
Code that breaks reveals what works.
ANALYTICAL REASONING RESULTS

MODEL: OpenAI GPT-4o-mini
Response time: 1.379 seconds
OUTPUT:
Utilizing multiple LLM providers enhances creativity, formatting reliability, and speed comparison, but introduces challenges in integration and testing complexity.

MODE

## Automatic Checks

These checks mirror the important formatting requirements in the starter assignment.

They do **not** decide which model is better; they only verify whether each response satisfies the explicit task constraints.


In [15]:
def creative_check(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return len(lines) >= 4


def analytical_check(text):
    text = text.strip()
    words = text.split()

    # Simple sentence count for this controlled task.
    sentence_endings = len(re.findall(r"[.!?](?:\s|$)", text))

    return (
        len(words) <= 40
        and sentence_endings == 1
        and text.endswith((".", "!", "?"))
    )


def instruction_check(text):
    try:
        data = json.loads(text)
    except Exception:
        return False

    if set(data.keys()) != {"todos"}:
        return False

    todos = data["todos"]
    if not isinstance(todos, list) or len(todos) != 2:
        return False

    for item in todos:
        if set(item.keys()) != {"task", "done"}:
            return False
        if not isinstance(item["task"], str):
            return False
        if not isinstance(item["done"], bool):
            return False

    return True


def build_check_table(results, checker):
    rows = []
    for model, result in results.items():
        passed = False if result["error"] else checker(result["output"])
        rows.append({
            "Model": model,
            "Seconds": result["seconds"],
            "Format check": "PASS" if passed else "FAIL",
        })
    return rows


In [16]:
import pandas as pd

summary = pd.DataFrame(
    build_check_table(creative_results, creative_check)
    + build_check_table(analytical_results, analytical_check)
    + build_check_table(instruction_results, instruction_check)
)

summary["Task"] = (
    ["Creative"] * len(MODEL_FUNCTIONS)
    + ["Analytical"] * len(MODEL_FUNCTIONS)
    + ["Instruction"] * len(MODEL_FUNCTIONS)
)

summary = summary[["Task", "Model", "Seconds", "Format check"]]
display(summary)


,Task,Model,Seconds,Format check
0,Creative,OpenAI GPT-4o-mini,1.649,PASS
1,Creative,Google Gemini 2.5 Flash,0.977,PASS
2,Creative,Anthropic Claude 3.5 Sonnet,2.336,PASS
3,Analytical,OpenAI GPT-4o-mini,1.379,PASS
4,Analytical,Google Gemini 2.5 Flash,0.767,PASS
5,Analytical,Anthropic Claude 3.5 Sonnet,1.366,PASS
6,Instruction,OpenAI GPT-4o-mini,0.838,PASS
7,Instruction,Google Gemini 2.5 Flash,0.771,PASS
8,Instruction,Anthropic Claude 3.5 Sonnet,1.563,PASS


## Manual Comparison Notes

After reviewing the actual outputs above, fill in the observations below. The notebook intentionally does not invent model-quality judgments; those should come from the playtest you actually ran.

| Task | Model observations |
|---|---|
| Creative Writing | Compare creativity, imagery, tone, originality, and 4-line compliance. |
| Analytical Reasoning | Compare correctness, conciseness, clarity, and one-sentence/40-word compliance. |
| Instruction Following | Compare JSON validity, exact schema compliance, and extra-text behavior. |
| Speed | Compare the measured response times shown above. |
| Reliability | Consider whether any request failed, timed out, or violated the requested format. |


## Reflection — 700–900 Words

The following cell creates a reflection draft from the **actual outputs and measured response times** after the notebook has been run. Read it, edit any wording that does not match your observations, and then paste the final version into your submission.


In [17]:
def first_sentence(text):
    match = re.search(r"(.+?[.!?])(?:\s|$)", text.strip())
    return match.group(1) if match else text.strip()


def reflection_draft():
    all_results = {
        "Creative Writing": creative_results,
        "Analytical Reasoning": analytical_results,
        "Instruction Following": instruction_results,
    }

    # Measured speed observations
    speed_lines = []
    for task, results in all_results.items():
        successful = [
            (model, data["seconds"])
            for model, data in results.items()
            if not data["error"]
        ]
        if successful:
            fastest_model, fastest_time = min(successful, key=lambda x: x[1])
            speed_lines.append(
                f"For {task.lower()}, the fastest measured response in this run was "
                f"{fastest_model} at {fastest_time:.3f} seconds."
            )

    draft = f"""### Reflection

This assignment gave me an opportunity to compare multiple large language models through a single API gateway instead of treating each provider as a completely separate integration. I used OpenRouter with the same API key and sent identical prompts to an OpenAI model, a Gemini model, and a Claude model. The three tasks were creative writing, analytical reasoning, and strict instruction following. Keeping the prompts identical was important because it made the model itself the main variable in the comparison.

For the creative-writing task, I compared how the models interpreted the request for a four-line poem about learning by testing code. The main differences I looked for were imagery, tone, originality, and whether the four-line constraint was respected. Creative tasks allow more freedom than analytical or structured tasks, so differences in wording and style are expected. My comparison should focus on the actual outputs rather than assuming that a particular provider is always more creative. In this run, the outputs should be judged by how effectively they communicate the theme while remaining concise and following the requested structure.

For the analytical task, the instruction was more constrained: summarize the supplied idea in one sentence with a maximum of forty words. This made accuracy, conciseness, and instruction compliance easier to evaluate. A useful response needs to preserve the central trade-off: using multiple providers can improve comparison and flexibility, but it also introduces additional integration and testing complexity. I also checked whether each model stayed within the word limit and produced exactly one sentence. This demonstrated that a model can be broadly capable while still needing careful prompting when an application requires a precise output format.

The instruction-following task was the most technical of the three because the output had to be valid JSON with exactly two todo items and specific data types. This resembles a common production use case in which an LLM response is passed to another program. Extra commentary or Markdown code fences can make an otherwise useful response fail parsing. I therefore checked the response using Python's JSON parser and verified the exact keys, item count, and data types. This showed why structured-output requirements should be validated programmatically instead of trusting a model response blindly.

API mechanics also influenced the experiment. OpenRouter provides an OpenAI-compatible endpoint, so the same Python client can send requests to different provider models by changing the model identifier. The important parameters included the model name, messages, temperature, and maximum output tokens. I kept temperature modest and limited the output length to control unnecessary verbosity, cost, and latency. The measured response time was also recorded for each call. The results demonstrate that model selection is not only about answer quality; latency and reliability matter when designing an application.

The experiment also suggests a practical approach to automatic model routing. Instead of sending every request to one model, an application could classify the incoming task first. Creative generation could be routed according to the application's desired writing style, structured tasks could be routed to a model that reliably follows schemas, and reasoning-heavy tasks could use a model that performs well on the application's evaluation set. The router should use measurable criteria such as latency, cost, format compliance, and task-specific evaluation results rather than assuming that one model is best for everything.

Overall, the main lesson is that a multi-model system provides flexibility but also creates engineering overhead. One API gateway can simplify provider access, but the application still has to manage model identifiers, token limits, errors, latency, output validation, and changing model availability. The experiment showed me that different models can respond differently to the same prompt, so model choice should be treated as an engineering decision based on the requirements of the task. A reliable production system would therefore combine model routing with automated evaluation and validation rather than depending on a single model for every type of request.

"""

    return draft

print(reflection_draft())


### Reflection

This assignment gave me an opportunity to compare multiple large language models through a single API gateway instead of treating each provider as a completely separate integration. I used OpenRouter with the same API key and sent identical prompts to an OpenAI model, a Gemini model, and a Claude model. The three tasks were creative writing, analytical reasoning, and strict instruction following. Keeping the prompts identical was important because it made the model itself the main variable in the comparison.

For the creative-writing task, I compared how the models interpreted the request for a four-line poem about learning by testing code. The main differences I looked for were imagery, tone, originality, and whether the four-line constraint was respected. Creative tasks allow more freedom than analytical or structured tasks, so differences in wording and style are expected. My comparison should focus on the actual outputs rather than assuming that a particular provide

## Final Submission Checklist

- [ ] Entered the OpenRouter API key using `getpass`; it is not hard-coded.
- [ ] Ran the setup and function cells without errors.
- [ ] Ran all three task comparisons.
- [ ] Included outputs from at least two models/providers.
- [ ] Reviewed the measured response times.
- [ ] Reviewed the automatic format checks.
- [ ] Edited the reflection so it accurately matches the actual outputs.
- [ ] Confirmed the reflection is between 700 and 900 words.
- [ ] Saved the completed notebook.
- [ ] Checked that the API key is not visible in any cell output or screenshot.
- [ ] Saved the notebook to GitHub if required by the LMS.
